# Proyecto 1 — Limpieza y preparación de datos

Este notebook revisa la calidad del dataset AnAge Animalia y guarda una copia preparada sin modificar los datos originales. Aquí solo se hace limpieza y preparación: no se interpretan resultados biológicos ni se responden las preguntas del análisis.

## 1. Importación de librerías

Usamos `pandas` para leer y preparar la tabla. `Path` sirve únicamente para encontrar los archivos del proyecto.

In [2]:
import pandas as pd
from pathlib import Path

## 2. Rutas del proyecto

La entrada está en `datos/raw` y la salida en `datos/procesados`. Si ejecutas el notebook desde la carpeta `notebooks`, se toma como base la carpeta superior del proyecto.

In [ ]:
ruta_proyecto = Path.cwd()
if ruta_proyecto.name == 'notebooks':
    ruta_proyecto = ruta_proyecto.parent

ruta_datos = ruta_proyecto / 'datos' / 'raw' / 'anage_animalia.csv'
ruta_salida = ruta_proyecto / 'datos' / 'procesados' / 'anage_animalia_preparado.csv'

print('Archivo de entrada:', ruta_datos)

## 3. Carga del archivo original

Leemos el CSV y hacemos una copia de trabajo. Así, los cambios posteriores no afectan los datos crudos.

In [ ]:
# Leer el archivo original y trabajar sobre una copia.
datos_originales = pd.read_csv(ruta_datos)

df = datos_originales.copy()

## 4. Inspección inicial

Antes de cambiar algo, comprobamos el tamaño de la tabla, sus columnas y sus tipos. Esta inspección ayuda a detectar problemas de lectura; todavía no modifica ningún dato.

In [ ]:
print('Filas y columnas:', df.shape)
print('Nombres de las columnas:')
print(df.columns.tolist())
display(df.head())
print('Tipos de datos:')
display(df.dtypes)
df.info()

## 5. Valores faltantes y valores únicos

`isna()` identifica celdas vacías; `sum()` las cuenta y `mean()` calcula la proporción por columna. `nunique()` cuenta los valores distintos sin contar los vacíos. El porcentaje permite comparar columnas aunque todas tengan el mismo número de filas.

Por ahora **no rellenamos ni eliminamos faltantes**: algunas mediciones son escasas y la decisión depende de las variables que necesite el equipo. `unknown`, cuando aparece como categoría, tampoco se convierte automáticamente en vacío.

In [ ]:
resumen_calidad = pd.DataFrame({
    'tipo': df.dtypes,
    'faltantes': df.isna().sum(),
    'porcentaje_faltantes': (df.isna().mean() * 100).round(2),
    'valores_unicos': df.nunique()
})

display(resumen_calidad.sort_values('porcentaje_faltantes', ascending=False))

## 6. Revisión de duplicados

Un duplicado exacto repite toda la fila. También revisamos si se repite `HAGRID`, que identifica registros, y la pareja `Genus` + `Species`. Repetir solo una clase o un género no sería un duplicado: muchas especies pueden compartirlos. No eliminamos filas solo por tener el mismo valor en una columna.

In [ ]:
duplicados_exactos = df.duplicated().sum()
identificadores_repetidos = df.duplicated(subset=['HAGRID']).sum()
especies_repetidas = df.duplicated(subset=['Genus', 'Species']).sum()

print('Filas duplicadas exactamente:', duplicados_exactos)
print('HAGRID repetidos:', identificadores_repetidos)
print('Parejas Genus + Species repetidas:', especies_repetidas)

## 7. Revisión de categorías principales

`value_counts()` permite ver las formas en que están escritas las categorías. Revisamos las columnas de clasificación y calidad antes de decidir si hace falta unificarlas. Conservamos mayúsculas y nombres científicos: cambiarlos podría alterar su significado.

In [ ]:
columnas_categoricas = ['Kingdom', 'Phylum', 'Class',
                        'Specimen origin', 'Sample size', 'Data quality']

for columna in columnas_categoricas:
    print('\n', columna)
    display(df[columna].value_counts(dropna=False))

## 8. Preparación inicial

Trabajamos sobre otra copia. `str.strip()` quita espacios al principio y al final de los textos, pero no cambia los espacios entre palabras ni las mayúsculas. Contamos cuántas celdas requerían ese ajuste; el `dropna()` del conteo solo omite vacíos temporalmente, no borra filas del archivo. `drop_duplicates()` quita únicamente filas completamente idénticas. Los faltantes se conservan porque rellenarlos con una media o mediana podría inventar mediciones de especies distintas.

In [ ]:
df_preparado = df.copy()
columnas_texto = df_preparado.select_dtypes(include='object').columns
espacios_sobrantes = 0

for columna in columnas_texto:
    textos_presentes = df_preparado[columna].dropna()
    espacios_sobrantes = espacios_sobrantes + (textos_presentes != textos_presentes.str.strip()).sum()
    df_preparado[columna] = df_preparado[columna].str.strip()

print('Celdas con espacios sobrantes corregidas:', espacios_sobrantes)

filas_antes = len(df_preparado)
df_preparado = df_preparado.drop_duplicates().copy()
print('Duplicados exactos retirados:', filas_antes - len(df_preparado))

## 9. Revisión de variables numéricas

Revisamos mínimos, máximos y valores no positivos en las mediciones. `HAGRID` se excluye porque es un identificador, no una medida. Un valor muy grande puede ser válido para alguna especie; esta revisión **no borra valores extremos** ni los interpreta.

In [ ]:
datos_numericos = df_preparado.select_dtypes(include='number').drop(columns='HAGRID')

print('Mínimos y máximos de las mediciones:')
display(datos_numericos.describe().loc[['min', 'max']])

print('Valores iguales o menores que cero por columna:')
display((datos_numericos <= 0).sum())

## 10. Validación y guardado

Comparamos el tamaño, los faltantes y los duplicados antes de guardar. El archivo preparado se escribe aparte, sin tocar `datos/raw`. Después lo leemos de nuevo para comprobar que conserva el mismo número de filas y columnas.

In [ ]:
print('Filas y columnas originales:', df.shape)
print('Filas y columnas preparadas:', df_preparado.shape)
print('Faltantes originales:', df.isna().sum().sum())
print('Faltantes preparados:', df_preparado.isna().sum().sum())
print('Duplicados exactos restantes:', df_preparado.duplicated().sum())
print('HAGRID repetidos restantes:', df_preparado.duplicated(subset=['HAGRID']).sum())

df_preparado.to_csv(ruta_salida, index=False)
datos_guardados = pd.read_csv(ruta_salida)
print('Tamaño del archivo guardado:', datos_guardados.shape)
print('Archivo preparado:', ruta_salida)

## Próximos pasos

Esta preparación conserva las 31 columnas y las mediciones faltantes. Cuando el equipo defina las preguntas concretas, podrá decidir si necesita tratar faltantes o casos extremos en una copia adicional, justificando el cambio y sin alterar el archivo original.